# OOP in Python
Python is a class-oriented language, which means that the main building blocks are classes and objects.

Basic principles:
* Class = template for creating objects
* Object = a specific instance of a class
* self = a reference to the current instance (equivalent to this in other languages)
* Everything is public – there is no private or protected, but the convention **_name** means "internal"
* Inheritance – classes can inherit from other classes (including multiple inheritance)
* Override – methods can be overridden in a descendant
* All classes automatically inherit from the base class **object**

## \_\_init\_\_ 

* the class initializer (constructor)
* A special method that is called automatically when an object is created.
* Used to set the initial state of the object, i.e. assign values to its attributes.
* The name must not be changed – it must always be \_\_init\_\_.
* The first parameter is self, which is a reference to the instance currently being created.
* If \_\_init\_\_ is not present in a class, Python creates the object with a default state and does not allow passing parameters at instantiation.

Definition of the Point class

In [ ]:
from math import sqrt
class Point:
    """ Point in 2d """
    
    def __init__(self, x, y):
        # constructor
        self.x = x
        self.y = y
        self._z = 3      # internal attribute, but it's just a convention; it's easy to access
        self.__w = 4     # private attribute; do not use it even in subclasses; it can be accessed there in a more complicated way via __dict__
        
    def distance (self, other):
        # method
        return sqrt((other.x - self.x) **2 + (other.y - self.y)**2)

Creating instances of points

In [ ]:
a = Point (1, 2)
b = Point (4, 5)
print (a.distance(b))   #  "a" is automatically substituted for "self"

# Inheritance
* Inheritance allows you to create a new class (a so-called descendant / subclass) that takes over the properties and methods of another class (a so-called parent / superclass).
* Used for code reuse and for organizing classes hierarchically.
* Python also supports multiple inheritance.
* **super** is used to call a method from the parent class.

In [ ]:
class A:
    def foo(self):
        print ("A.foo()")
        
class B(A):
    def foo(self):
        super().foo()        
        print ("B.foo()")

class C(A):
    def foo(self):
        print ("C.foo()")
        
class D(B, C):
    def bar(self):
        print ("D.bar()")

In [ ]:
b=B()
b.foo()

The inheritance hierarchy can be printed using **\_\_base\_\_**.

In [ ]:
D.__bases__

* Python uses MRO (Method Resolution Order) – the order in which it looks for methods in multiple inheritance.
* It is implemented as a linearized list of the class inheritance hierarchy.
* The method from the first class found is called.

In [ ]:
d=D()
d.foo()
print (D.__mro__)

## Instance and class variables
Python allows two main levels of variables in a class.

* **Instance variables** belong to a specific object (instance).
    * Each object has its own copy of these variables.
    * Defined inside the __init__ method using self.

* **Class variables** belong to the class itself, and are shared by all instances.
    * Defined directly in the class, outside __init__.
    * Access from an instance: self.name (if the instance doesn't have the same attribute)
    * Access from the class: Class.name

In [ ]:
class E:
    x = 1           # class variable
    def __init__(self, y):
        self.y=y    # instance variable

In [ ]:
e1 = E(5)
e2 = E(3)
print (E.x, e1.x, e1.y)
print (E.x, e2.x, e1.y)

In [ ]:
e1.x=2
print (E.x, e1.x, e1.y)
print (E.x, e2.x, e1.y)

In [ ]:
E.x=2
print (E.x, e1.x, e1.y)
print (E.x, e2.x, e1.y)

## Static and class methods
In Python, besides instance methods, there are two special categories of methods.

* **Instance method** is a standard class method.
    * The first parameter is always **self**, which is a reference to the specific instance.
    * Can access both instance and class variables.

* **Class method** (@classmethod)
    * The first parameter is **cls**, which refers to the class, not the instance.
    * Can only access class variables and other class methods.
    * Decorator: @classmethod    

* **Static method** (@staticmethod)
    * Has no automatic self or cls parameter.
    * Works like a regular function, but is part of the class.
    * Has no access to instance or class variables, but logically belongs to the class.
    * Decorator: @staticmethod    

In [ ]:
class Car:
    number_of_wheel = 4  # class variable

    def __init__(self, color):
        self.color = color  # instance variable

    def info(self):
        print(f"Car has {self.number_of_wheel} wheels and color is {self.color}.")

    @classmethod
    def info_wheels(cls):
        print(f"All cars has {cls.number_of_wheel} wheels.")        

    @staticmethod
    def avg_speed(distance, time):
        return distance / time        

a = Car("red")
a.info()  
Car.info_wheels()
print(Car.avg_speed(120, 2))

## Creating instances

* **\_\_new\_\_** - the method that creates the instance itself.
   * Called before \_\_init\_\_
   * The first parameter is cls, a reference to the class the instance will be created for.
   * Typically only used in advanced patterns (e.g. Singleton).
   * Returns a new instance of the object (return super().\_\_new\_\_(cls)).
* **\_\_init\_\_** - the initialization method

In [ ]:
class G:
    def __new__ (cls, x):
        print ('G.__new__()')
        return object.__new__(cls)
    
    def __init__(self, x):
        print ('G.__init__()')
        self.x = x

In [ ]:
g=G(1)

## Storage in memory
In Python, all instance variables of an object are stored in an internal dictionary.

* Each object has its own dictionary accessible via __dict__.
* Key = attribute name, value = reference to the object (data value).
* Very convenient from a programmer's point of view.
* However, dictionaries take up more memory.
* Every access to a variable requires a lookup in a hash table.

In [ ]:
e1.__dict__

In [ ]:
e1.z="abc"

In [ ]:
e1.__dict__

## Optimization
If we want to save memory for classes with a large number of instances, we can use __slots__:
* An instance cannot have attributes other than those defined in slots.
* Implemented in memory as an array, accessed via indexes.
* Less memory-intensive.
* We lose dynamism.

In [ ]:
class H:
    __slots__ = ('x', 'y')
    def __init__ (self, x, y):
        self.x = x
        self.y = y

In [ ]:
h=H(1, 4)

In [ ]:
# will result in an error
h.z=3

## Special methods
* In Python, we have several ways to control access to an object's attributes and to release resources.
* Python recommends direct access to attributes (object.attribute), but sometimes we need control over reading or writing.
* The **@property** decorator and the additional **@<attribute>.setter** are used for this.

In [ ]:
class Person:
    def __init__(self, name, age):
        self._name = name      # underscore = "private" attribute convention
        self._age = age

    # getter
    @property
    def age(self):
        return self._age

    # setter
    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("Age cannot be negative.")
        self._age = value

o = Person("Eva", 25)
print(o.age)
o.age = 30
print(o.age)
o.age = -5

The special method **\_\_del\_\_** is called when an object is deleted and released by the garbage collector.

It is used to release resources (files, sockets, connections, etc.).

In [ ]:
class File:
    def __init__(self, name):
        self.name = name
        self.f = open(name, "w")

    def __del__(self):
        print(f"Closing file {self.name}")
        self.f.close()

s = File("test.txt")
del s 

The special methods **\_\_str\_\_** and **\_\_repr\_\_** control how the object is displayed as a string.

* `__str__` — a readable printout for the user, called by `print(object)` or `str(object)`
* `__repr__` — an exact representation for developers, called in the interactive environment or by `repr(object)`. It should ideally contain enough information to reconstruct the object.

Without these methods, Python only prints `<__main__.Point object at 0x...>`.

In [ ]:
class Point2:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"Point({self.x}, {self.y})"

    def __repr__(self):
        return f"Point2(x={self.x}, y={self.y})"

p = Point2(3, 4)
print(p)        # call __str__  →  Point(3, 4)
print(repr(p))  # call __repr__ →  Point2(x=3, y=4)

points = [Point2(1, 2), Point2(5, 6)]
print(points)   # The list calls __repr__ on its elements

# Exercise 1
Create an Animal class with instance variables:
- name (str)
- age (int)

Add an info() method that prints: Name: \<name\>, Age: \<age\>

In [ ]:
# code

In [ ]:
z = Animal("lion", 5)
z.info()

# Exercise 2
Add a class variable number_animals = 0

Each new Animal instance increases number_animals by 1

In [ ]:
# code

In [ ]:
z1 = Animal("lion", 3)
z2 = Animal("tiger", 4)
print(Animal.number_animals)

# Exercise 3
Create a Bird class that inherits from Animal

Add an instance variable type (e.g. "Parrot")

Override the info() method so that it prints: Name: \<name\>, Age: \<age\>, Type: \<type\>

In [ ]:
# code

In [ ]:
z3 = Bird("bird", 50, "parrot")
z3.info()

# Exercise 4
Add a getter and setter for the Animal class's _age attribute using @property and @setter.

The setter should check that the age is not negative, otherwise it raises a ValueError.

In [ ]:
# code

In [ ]:
z4 = Animal("Elephant", 10)
z4.age = -5 

# Exercise 5
Add a \_\_del\_\_() method to the Animal class that prints: Releasing animal \<name\>

In [ ]:
# code

In [ ]:
z5 = Animal ("Cat", 30)
del z5

# Exercise 6

Add a static method that calculates the average age of a list of animals.

Add a class method that prints the total number of animals.

In [ ]:
# Define the static method average_age(animals) and the class method print_count()


In [ ]:
z6a = Animal("Dog", 4)
z6b = Animal("Cat", 7)
z6c = Animal("Fox", 3)

print(Animal.average_age([z6a, z6b, z6c]))   # 4.666...
Animal.print_count()